# GBM 모델 성능 개선 V2
## GBM만 사용
- 추가 파생 변수 (row_mean, row_median, row_max, row_min, row_range, n_negative)
- GBM 기반 피처 선택
- 5-Fold 교차검증
- 조기 중단 최적화

In [11]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier

print('라이브러리 로드 완료')

라이브러리 로드 완료


In [12]:
DATA_PATH = os.path.join('../../data/', 'santander-customer-satisfaction')
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))

y_labels = train_df.iloc[:, -1].copy()
X_features = train_df.drop(columns=['ID', 'TARGET']).copy()

print(f'데이터 크기: {X_features.shape}')
print(f'클래스 분포:')
print(y_labels.value_counts())

데이터 크기: (76020, 369)
클래스 분포:
TARGET
0    73012
1     3008
Name: count, dtype: int64


In [13]:
X_features['var3'] = X_features['var3'].replace(-999999, 2)
X_features['var38'] = np.log1p(X_features['var38'])

X_features['n0'] = (X_features == 0).sum(axis=1)
X_features['row_std'] = X_features.std(axis=1)
X_features['row_mean'] = X_features.mean(axis=1)
X_features['row_median'] = X_features.median(axis=1)
X_features['row_max'] = X_features.max(axis=1)
X_features['row_min'] = X_features.min(axis=1)
X_features['row_range'] = X_features['row_max'] - X_features['row_min']
X_features['n_negative'] = (X_features < 0).sum(axis=1)

print('기본 전처리 및 파생 변수 생성 완료')

기본 전처리 및 파생 변수 생성 완료


In [14]:
zero_var_cols = [col for col in X_features.columns if X_features[col].nunique() == 1]
X_features.drop(columns=zero_var_cols, inplace=True)
print(f'제거된 상수 컬럼: {len(zero_var_cols)}')

dup_cols = X_features.T.duplicated()
dup_col_names = X_features.columns[dup_cols].tolist()
X_features.drop(columns=dup_col_names, inplace=True)
print(f'제거된 중복 컬럼: {len(dup_col_names)}')

manual_remove = [c for c in X_features.columns if 'var6' in c] + [
    'delta_imp_reemb_var13_1y3', 'delta_imp_reemb_var17_1y3', 
    'delta_imp_trasp_var17_in_1y3', 'delta_imp_trasp_var33_in_1y3'
]
manual_remove = [c for c in manual_remove if c in X_features.columns]
X_features.drop(columns=manual_remove, inplace=True)
print(f'제거된 노이즈 컬럼: {len(manual_remove)}')
print(f'현재 피처 수: {X_features.shape[1]}')

제거된 상수 컬럼: 35
제거된 중복 컬럼: 29
제거된 노이즈 컬럼: 9
현재 피처 수: 304


In [15]:
print('GBM으로 중요 피처 선택 중...')

gbm_selector = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=156
)
gbm_selector.fit(X_features, y_labels)

feat_imp = pd.Series(gbm_selector.feature_importances_, index=X_features.columns)
zero_imp_cols = feat_imp[feat_imp == 0].index.tolist()

X_features = X_features.drop(columns=zero_imp_cols)
print(f'제거된 중요도 0 피처: {len(zero_imp_cols)}')
print(f'최종 피처 수: {X_features.shape[1]}')

GBM으로 중요 피처 선택 중...
제거된 중요도 0 피처: 208
최종 피처 수: 96


In [16]:
print('Scaling & PCA 학습 중...')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

pca = PCA(n_components=5, random_state=156)
X_pca = pca.fit_transform(X_scaled)

X_final = X_features.copy()
for i in range(5):
    X_final[f'pca{i+1}'] = X_pca[:, i]

print(f'최종 데이터셋: {X_final.shape}')

Scaling & PCA 학습 중...
최종 데이터셋: (76020, 101)


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_labels,
    test_size=0.2,
    stratify=y_labels,
    random_state=0
)

print(f'훈련 데이터: {X_train.shape}')
print(f'테스트 데이터: {X_test.shape}')

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
print(f'클래스 0: {n_neg}, 클래스 1: {n_pos}')

훈련 데이터: (60816, 101)
테스트 데이터: (15204, 101)
클래스 0: 58410, 클래스 1: 2406


In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=156)

cv_results = []
fold_models = []

print('=' * 60)
print('5-Fold 교차검증 시작')
print('=' * 60)

for fold_num, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_val = y_train.iloc[val_idx]
    
    gbm = GradientBoostingClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        max_features='sqrt',
        validation_fraction=0.2,
        n_iter_no_change=30,
        tol=1e-4,
        random_state=156
    )
    
    gbm.fit(X_tr, y_tr)
    
    val_pred = gbm.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_pred)
    
    cv_results.append(val_auc)
    fold_models.append(gbm)
    
    print(f'Fold {fold_num}: 검증 AUC = {val_auc:.4f}')

print('=' * 60)
print(f'평균 검증 AUC: {np.mean(cv_results):.4f} (±{np.std(cv_results):.4f})')
print('=' * 60)

5-Fold 교차검증 시작
Fold 1: 검증 AUC = 0.8384
Fold 2: 검증 AUC = 0.8397
Fold 3: 검증 AUC = 0.8487
Fold 4: 검증 AUC = 0.8352
Fold 5: 검증 AUC = 0.8438
평균 검증 AUC: 0.8412 (±0.0047)


In [19]:
print('\n최종 테스트 세트 평가')
print('=' * 60)

test_auc_scores = []
test_pred_ensemble = np.zeros(len(X_test))

for fold_num, model in enumerate(fold_models, 1):
    test_pred = model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_pred)
    test_auc_scores.append(test_auc)
    test_pred_ensemble += test_pred
    
    print(f'Fold {fold_num}: 테스트 AUC = {test_auc:.4f}')

test_pred_ensemble /= len(fold_models)
ensemble_auc = roc_auc_score(y_test, test_pred_ensemble)

print('=' * 60)
print(f'개별 평균 테스트 AUC: {np.mean(test_auc_scores):.4f}')
print(f'앙상블 테스트 AUC: {ensemble_auc:.4f}')
print('=' * 60)


최종 테스트 세트 평가
Fold 1: 테스트 AUC = 0.8235
Fold 2: 테스트 AUC = 0.8238
Fold 3: 테스트 AUC = 0.8229
Fold 4: 테스트 AUC = 0.8195
Fold 5: 테스트 AUC = 0.8219
개별 평균 테스트 AUC: 0.8223
앙상블 테스트 AUC: 0.8246


In [20]:
print('\n' + '=' * 60)
print('최종 결과')
print('=' * 60)
print(f'기존 GBM 테스트 AUC: 0.8188')
print(f'개선 GBM 테스트 AUC: {ensemble_auc:.4f}')
print(f'개선율: {(ensemble_auc - 0.8188) / 0.8188 * 100:+.2f}%')
print('=' * 60)

print('\n적용된 개선사항:')
print('  1. 추가 파생 변수 (행 통계량 확장)')
print('  2. GBM 기반 피처 선택')
print('  3. Scaling + PCA 적용')
print('  4. 5-Fold 교차검증')
print('  5. 조기 중단 최적화')
print('  6. 폴드 앙상블 (평균 예측확률)')


최종 결과
기존 GBM 테스트 AUC: 0.8188
개선 GBM 테스트 AUC: 0.8246
개선율: +0.70%

적용된 개선사항:
  1. 추가 파생 변수 (행 통계량 확장)
  2. GBM 기반 피처 선택
  3. Scaling + PCA 적용
  4. 5-Fold 교차검증
  5. 조기 중단 최적화
  6. 폴드 앙상블 (평균 예측확률)
